## Write a solution to find the employees who earn more than their managers.

## Return the result table in any order.

## The result format is in the following example.

 

Example 1:

Input: 
Employee table:
+----+-------+--------+-----------+
| id | name  | salary | managerId |
+----+-------+--------+-----------+
| 1  | Joe   | 70000  | 3         |
| 2  | Henry | 80000  | 4         |
| 3  | Sam   | 60000  | Null      |
| 4  | Max   | 90000  | Null      |
+----+-------+--------+-----------+
Output: 
+----------+
| Employee |
+----------+
| Joe      |
+----------+

## taking pandas from leet codes and coverting pandas to spark data frame

In [1]:
import pandas as pd
data = [[1, 'Joe', 70000, 3], [2, 'Henry', 80000, 4], [3, 'Sam', 60000, None], [4, 'Max', 90000, None]]
employee = pd.DataFrame(data, columns=['id', 'name', 'salary', 'managerId']).astype({'id':'Int64', 'name':'object', 'salary':'Int64', 'managerId':'Int64'})


In [2]:
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName("Emp earnig more than Manager").getOrCreate()
df=spark.createDataFrame(employee)
df.show()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/18 06:10:15 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


+---+-----+------+---------+
| id| name|salary|managerId|
+---+-----+------+---------+
|  1|  Joe| 70000|      3.0|
|  2|Henry| 80000|      4.0|
|  3|  Sam| 60000|      NaN|
|  4|  Max| 90000|      NaN|
+---+-----+------+---------+



26/06/18 06:10:26 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [7]:
from pyspark.sql.functions import col
df1=df.alias("emp").join(df.alias("mgr"), col("emp.managerId") == col("mgr.id"), "left")\
.select(col("emp.name").alias("Employee"))
df2=df1.filter(col("emp.salary") > col("mgr.salary"))

df2.show()


+--------+
|Employee|
+--------+
|     Joe|
+--------+



## creating temp view and writing sql for the same question

In [11]:
df.createOrReplaceTempView("EmpMan")
sqlDF=spark.sql("""
                 select a.name as Employee 
                from EmpMan a
                 join EmpMan b
                on a.managerId=b.id
                where a.salary>b.salary
                """)
sqlDF.show()

+--------+
|Employee|
+--------+
|     Joe|
+--------+



26/06/18 10:49:36 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 15185791 ms exceeds timeout 120000 ms
26/06/18 10:49:36 WARN SparkContext: Killing executors is not supported by current scheduler.
26/06/18 10:49:36 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint